# Test Fine-tuning Results
Test the fine-tuned LIANet Creoss region performance to the local model perfomance

In [1]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from helpers import load_model_class
import sys
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from rasterio.windows import Window
# Add src to path
sys.path.insert(0, '/home/user/src')
import rasterio as rio
from datasets import BuildingCoverageRaster
from models.models_finetune import UNet, MicroUNet
from settings import *



from torchmetrics import MetricCollection

from metrics import multiclass_segmentation_metrics
# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
import glob
from models.models_finetune import DownstreamModel
from models.LIANet import LIANetLight
import omegaconf, hydra

pretrained_model_path = {"BFPDensity_joint_T31TFM": "/home/user/results_shared/fourier_learned_4regions/2026-03-18_19-06-01",
                         "BFPDensity_joint_T32ULU": "/home/user/results_shared/fourier_learned_4regions/2026-03-18_19-06-01",
                         "BFPDensity_local_T31TFM": "/home/user/results_shared/fourier_learned_T31TFM/2026-04-08_00-13-56",
                         "BFPDensity_local_T32ULU": "/home/user/results_shared/fourier_learned_T32ULU/2026-04-04_09-21-54"}


def load_model(CKPT_PATH, other_task, model_type):
    model_finetune = load_model_class(
        other_task, 
        model_type, 
        MODEL_PATH= pretrained_model_path[other_task],
        NUM_CLASSES=num_classes[other_task],
        ACTIVATION_FUNCTION=activation_functions[other_task]
    )
    checkpoint = torch.load(CKPT_PATH, map_location=device)
    state_dict = checkpoint["model_state_dict"]
    if state_dict and all(k.startswith("module.") for k in state_dict.keys()):
        state_dict = {k[len("module."):]: v for k, v in state_dict.items()}

    model_finetune.load_state_dict(state_dict, strict=True)

    model_finetune = model_finetune.to(device)
    model_finetune.eval()
    # print("✓ Model loaded successfully")
    return model_finetune

In [ ]:
import os
import torch
import pandas as pd

from tqdm import tqdm
from torchmetrics import MetricCollection
from metrics import regression_metrics

model_type_list = ["replace_final_block", "unet", "micro_unet"]

for model_type in model_type_list:

    if model_type == "replace_final_block":
        all_tasks_list = {
            "BFPDensity_joint_T31TFM": ["BFPDensity_joint_T32ULU"],
            "BFPDensity_joint_T32ULU": ["BFPDensity_joint_T31TFM"],
            "BFPDensity_local_T31TFM": ["BFPDensity_local_T31TFM"],
            "BFPDensity_local_T32ULU": ["BFPDensity_local_T32ULU"],
        }
    elif model_type == "unet" or model_type == "micro_unet":
        all_tasks_list = {
            "BFPDensity_local_T31TFM": ["BFPDensity_local_T32ULU"],
            "BFPDensity_local_T32ULU": ["BFPDensity_local_T31TFM"],
        }

    BATCH_SIZE = 16
    NUM_WORKERS = 8

    results_rows = []

    for Target_region in all_tasks_list:
        application_name = Target_region.split("_")[0]
        other_tasks = all_tasks_list[Target_region]
        
        # Dataset of the Target region for evaluation
        val_dataset = BuildingCoverageRaster(
            top_dir=TOP_DIR[Target_region],
            s2_tiles=s2_tiles[Target_region],
            labels=labels[Target_region],
            train_val_key="val",
        )

        dataloader = torch.utils.data.DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            drop_last=False,
        )

        for source_region in other_tasks:
            startername = "LIANet" if "replace_final_block" in model_type else "microunet" if "micro_unet" in model_type else "unet"
            model_dir = glob.glob(f"/home/user/results_local/finetuning_results_BFPDensity/{source_region}/{startername}*")
            # model_dir = (
            #     f"/home/user/results_local/finetuning_results_BFPDensity/{source_region}/LIANet_lr3e-05_batchsize8_nonburned"
            # )

            for run_name in sorted(os.listdir(model_dir[0])):
                ckpt_path = os.path.join(model_dir[0], run_name, "last.pt")

                if not os.path.exists(ckpt_path):
                    print(f"Model directory not found: {model_dir}. Skipping...")
                    continue

                model = load_model(ckpt_path, source_region, model_type)
                model.eval()

                if "BFPDensity" in source_region:
                    list_of_metrics, _ = regression_metrics()
                else:
                    list_of_metrics, _ = multiclass_segmentation_metrics()

                metrictracker = MetricCollection(list_of_metrics).to(device)

                with torch.no_grad():
                    for batch in tqdm(
                        dataloader,
                        desc=f"{Target_region} | {source_region} | {run_name}",
                        leave=False,
                    ):
                        x = batch["x_s2"].to(device)
                        y = batch["y_s2"].to(device)
                        label = batch["label"].to(device)
                        delta_days = batch["delta_days"].to(device)
                        target_tile = Target_region.split("_")[-1]
                        s2data  = batch["s2data"].to(device)
                        if "local" in Target_region:
                            region_idx = 0
                        elif "BFP" in Target_region:
                            region_idx = 1 if target_tile == "T32ULU" else 2 if target_tile == "T31TFM" else None
                        elif "joint" and "PASTIS" in Target_region:
                            region_indx = 0 if target_tile == "T31TFJ" else 1 if target_tile == "T32ULU" else 2 if target_tile == "T31TFM" else 3
                        elif "joint" and "BurnScars" in Target_region:
                            region_idx = 0 if target_tile == "T11SMT" else 1 if target_tile == "T16REV" else None
                        # print(f"Region idx for target tile {target_tile} is {region_idx}")
                        if "replace_final_block" in model_type:
                            _, pred = model(
                                delta_days,
                                x,
                                y,
                                torch.tensor([region_idx], device=device),
                            )
                        elif "micro_unet" in model_type or "unet" in model_type:
                            pred = model(s2data)

                        pred = getattr(pred, "output", pred)

                        if pred.dim() == 3:
                            pred = pred.unsqueeze(1)
                        # print(f'shape of pred: {pred.shape}, shape of label: {label.shape}')
                        metrictracker.update(pred.squeeze(1), label)

                results = metrictracker.compute()

                row = {
                    "model_type": model_type,
                    "target_region": Target_region,
                    "source_region": source_region,
                    "seed_or_run": run_name,
                    "checkpoint_path": ckpt_path,
                }

                for metric_name, metric_value in results.items():
                    row[metric_name] = float(metric_value.detach().cpu())

                results_rows.append(row)

                del model
                del metrictracker
                torch.cuda.empty_cache()

        del dataloader
        del val_dataset
        torch.cuda.empty_cache()


    df_results = pd.DataFrame(results_rows)

    df_results.to_csv(
        "{}_results_all_target_regions.csv".format(model_type),
        index=False,
    )

    

306 val samples loaded from /home/user/data_shared/T31TFM_val_samples_10perc.json


BFPDensity_local_T31TFM | BFPDensity_local_T32ULU | 2026-05-12_21-21-44:   0%|          | 0/20 [00:00<?, ?it/s]

513 val samples loaded from /home/user/data_shared/T32ULU_val_samples_10perc.json


In [17]:
df_results

,target_region,source_region,seed_or_run,checkpoint_path,mae,mse
0,BFPDensity_local_T31TFM,BFPDensity_local_T32ULU,2026-05-12_17-05-26,/home/user/results_local/finetuning_results_BF...,0.099279,0.041582
1,BFPDensity_local_T31TFM,BFPDensity_local_T32ULU,2026-05-12_17-08-00,/home/user/results_local/finetuning_results_BF...,0.098252,0.041648
2,BFPDensity_local_T31TFM,BFPDensity_local_T32ULU,2026-05-12_17-10-31,/home/user/results_local/finetuning_results_BF...,0.100289,0.042800
3,BFPDensity_local_T31TFM,BFPDensity_local_T32ULU,2026-05-12_17-13-07,/home/user/results_local/finetuning_results_BF...,0.100608,0.042893
4,BFPDensity_local_T31TFM,BFPDensity_local_T32ULU,2026-05-12_17-15-39,/home/user/results_local/finetuning_results_BF...,0.098162,0.041027
5,BFPDensity_local_T32ULU,BFPDensity_local_T31TFM,2026-05-12_17-05-23,/home/user/results_local/finetuning_results_BF...,0.103576,0.047076
6,BFPDensity_local_T32ULU,BFPDensity_local_T31TFM,2026-05-12_17-07-34,/home/user/results_local/finetuning_results_BF...,0.102534,0.046801
7,BFPDensity_local_T32ULU,BFPDensity_local_T31TFM,2026-05-12_17-09-42,/home/user/results_local/finetuning_results_BF...,0.104288,0.047767
8,BFPDensity_local_T32ULU,BFPDensity_local_T31TFM,2026-05-12_17-11-52,/home/user/results_local/finetuning_results_BF...,0.103346,0.047456
9,BFPDensity_local_T32ULU,BFPDensity_local_T31TFM,2026-05-12_17-14-05,/home/user/results_local/finetuning_results_BF...,0.102728,0.046731


In [6]:
import pandas as pd

model_type_list = ["replace_final_block", "unet", "micro_unet"]

avg_results = {}

for model_type in model_type_list:

    csv_path = f"{model_type}_results_all_target_regions.csv"

    print(f"\n========== {model_type} ==========")

    df_results = pd.read_csv(csv_path)

    df_results["training_type"] = (
        df_results["target_region"]
        .str.extract(r"_(joint|local)_")
    )

    metadata_cols = [
        "model_type",
        "target_region",
        "source_region",
        "seed_or_run",
        "checkpoint_path",
        "training_type",
    ]

    metric_cols = [
        col for col in df_results.select_dtypes(include="number").columns
        if col not in metadata_cols
    ]

    df_seed_avg = (
        df_results
        .groupby(
            ["training_type", "target_region", "source_region"],
            as_index=False
        )[metric_cols]
        .mean()
    )

    df_joint_local_avg = (
        df_seed_avg
        .groupby("training_type", as_index=False)[metric_cols]
        .mean()
    )

    avg_results[model_type] = df_joint_local_avg

    print(df_joint_local_avg)


========== replace_final_block ==========
  training_type       mae       mse
0         joint  0.133155  0.055639
1         local  0.127768  0.052510

========== unet ==========
  training_type       mae       mse
0         local  0.101306  0.044578

========== micro_unet ==========
  training_type       mae       mse
0         local  0.133873  0.066138
